In [2]:
from pyspark.sql import SparkSession

PACKAGES = [
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "org.postgresql:postgresql:42.6.0",
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
]

spark = SparkSession.builder \
    .appName("Permormance_Analytics") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.jars.packages", ",".join(PACKAGES)) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog") \
    .config("spark.sql.catalog.my_catalog.uri", "jdbc:postgresql://postgres:5432/iceberg_metastore") \
    .config("spark.sql.catalog.my_catalog.jdbc.user", "iceberg") \
    .config("spark.sql.catalog.my_catalog.jdbc.password", "iceberg") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://warehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
print('123')

123


In [3]:
import time
import os 
from datetime import datetime

LOG_READ_FILE = "benchmark_read_results_sorted.csv"

# create a file for read metrics if it does not exist
if not os.path.exists(LOG_READ_FILE):
    with open(LOG_READ_FILE, "w") as f:
        f.write("timestamp,table_name,files_count,execution_time_agg,execution_time_single\n")
    print(f"read log file created: {LOG_READ_FILE}")

def benchmark_read(table_name):
    current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"\n--- testing read: {table_name} [{current_time}] ---")
    
    # 1. heavy aggregation
    start_time_agg = time.time()
    spark.sql(f"""
        SELECT 
            shop,
            device_type,
            event_type,
            COUNT(DISTINCT session_id) as unique_sessions, 
            COUNT(DISTINCT user_id) as unique_users,
            SUM(price) as total_revenue, 
            AVG(price) as avg_check,
            MIN(timestamp) as first_interaction,
            MAX(timestamp) as latest_interaction
        FROM {table_name}
        WHERE shop IN ('rozetka', 'foxtrot', 'citrus') AND device_type IN ('ios', 'android')
        GROUP BY 1,2,3
        ORDER BY total_revenue DESC, unique_sessions DESC
    """).collect()
    execution_time_agg = time.time() - start_time_agg
    
    # 2. file count
    files_count = spark.sql(f"SELECT COUNT(*) FROM {table_name}.files").collect()[0][0]

    # 3. point query
    start_time_single = time.time()
    spark.sql(f"""
        SELECT * FROM {table_name}
        WHERE shop = 'rozetka' 
          AND device_type = 'mobile_web' 
          AND event_type = 'purchase'
    """).collect()
    execution_time_single = time.time() - start_time_single

    # 4. write to csv
    with open(LOG_READ_FILE, "a") as f:
        f.write(f"{current_time},{table_name},{files_count},{execution_time_agg:.4f},{execution_time_single:.4f}\n")
    
    print(f"files: {files_count}")
    print(f"agg (heavy query): {execution_time_agg:.4f} sec")
    print(f"single (point query): {execution_time_single:.4f} sec")

# example call:
# benchmark_read("my_catalog.default.user_events_unsorted")
# benchmark_read("my_catalog.default.user_events_sorted")

read log file created: benchmark_read_results_sorted.csv


In [5]:

INTERVAL_MINUTES = 30

try:
    while True:
        benchmark_read("my_catalog.default.user_events_unsorted")
        benchmark_read("my_catalog.default.user_events_sorted")
        print(f"waiting set interval")
        time.sleep(INTERVAL_MINUTES * 60)
except KeyboardInterrupt:
    print("interrupted")


--- testing read: my_catalog.default.user_events_unsorted [2026-05-21 06:05:13] ---
files: 1285
agg (heavy query): 9.8336 sec
single (point query): 5.2897 sec

--- testing read: my_catalog.default.user_events_sorted [2026-05-21 06:05:30] ---
files: 153587
agg (heavy query): 145.5003 sec
single (point query): 4.4519 sec
waiting set interval
interrupted
